In [30]:
import pandas as pd
import numpy as np

# Load without header
df = pd.read_csv("hepatitis.csv", header=None)

# Assign proper column names
df.columns = [
    'CLASS','AGE','SEX','STEROID','ANTIVIRALS','FATIGUE','MALAISE',
    'ANOREXIA','LIVER_BIG','LIVER_FIRM','SPLEEN_PALPABLE','SPIDERS',
    'ASCITES','VARICES','BILIRUBIN','ALK_PHOSPHATE','SGOT',
    'ALBUMIN','PROTIME','HISTOLOGY'
]

print(df.head())
print("Shape:", df.shape)
print(df['CLASS'].value_counts())

   CLASS  AGE  SEX STEROID  ANTIVIRALS FATIGUE MALAISE ANOREXIA LIVER_BIG  \
0      2   30    2       1           2       2       2        2         1   
1      2   50    1       1           2       1       2        2         1   
2      2   78    1       2           2       1       2        2         2   
3      2   31    1       ?           1       2       2        2         2   
4      2   34    1       2           2       2       2        2         2   

  LIVER_FIRM SPLEEN_PALPABLE SPIDERS ASCITES VARICES BILIRUBIN ALK_PHOSPHATE  \
0          2               2       2       2       2      1.00            85   
1          2               2       2       2       2      0.90           135   
2          2               2       2       2       2      0.70            96   
3          2               2       2       2       2      0.70            46   
4          2               2       2       2       2      1.00             ?   

  SGOT ALBUMIN PROTIME  HISTOLOGY  
0   18     4.0      

In [31]:
import pandas as pd
import numpy as np

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Convert to numeric
df = df.apply(pd.to_numeric, errors='coerce')

# Fill missing values (DO NOT drop rows)
df.fillna(df.mean(), inplace=True)

print("After Cleaning:", df.shape)
print(df['CLASS'].value_counts())

After Cleaning: (155, 20)
CLASS
2    123
1     32
Name: count, dtype: int64


In [32]:
# Only numeric continuous columns (safe for IQR)
cols = ['AGE', 'BILIRUBIN', 'ALK_PHOSPHATE', 'SGOT', 'ALBUMIN', 'PROTIME']
Q1 = df[cols].quantile(0.25)
Q3 = df[cols].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Clip values instead of removing rows
df[cols] = df[cols].clip(lower=lower, upper=upper, axis=1)

print("After Outlier Capping:", df.shape)
print(df['CLASS'].value_counts())

After Outlier Capping: (155, 20)
CLASS
2    123
1     32
Name: count, dtype: int64


In [33]:
from sklearn.preprocessing import StandardScaler

# Target variable (CLASS: 1 = DIE, 2 = LIVE)
y = df['CLASS']

# Convert to binary (0 = DIE, 1 = LIVE)
y = y.apply(lambda x: 1 if x == 2 else 0)

# Features
X = df.drop('CLASS', axis=1)

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

In [36]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", acc_lr)

Logistic Regression Accuracy: 0.7419354838709677


In [37]:
nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)

print("Naïve Bayes Accuracy:", acc_nb)

Naïve Bayes Accuracy: 0.6774193548387096


In [38]:
print("\nModel Comparison:")
print("Logistic Regression:", acc_lr)
print("Naïve Bayes:", acc_nb)

if acc_lr > acc_nb:
    print("✅ Logistic Regression performs better")
else:
    print("✅ Naïve Bayes performs better")


Model Comparison:
Logistic Regression: 0.7419354838709677
Naïve Bayes: 0.6774193548387096
✅ Logistic Regression performs better
